In [28]:
import torch
from torch import nn
from torchrl.envs import PettingZooWrapper
from env_simplified import MahjongGameEnv
from torchrl.envs import TransformedEnv
from torchrl.envs.transforms import ActionMask
from torchrl.envs.utils import MarlGroupMapType
from torchrl.modules import MultiAgentConvNet, MultiAgentMLP, ProbabilisticActor, MaskedCategorical
from tensordict.nn import TensorDictModule
from torchrl.collectors import Collector
from torchrl.data.replay_buffers import ReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.objectives import ClipPPOLoss, ValueEstimators

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

env = PettingZooWrapper(
    env=MahjongGameEnv(),
    use_mask=True,
    return_state=True,
    categorical_actions=True,
    group_map=MarlGroupMapType.ALL_IN_ONE_GROUP
)


cuda


c:\Users\ctc73\PycharmProjects\MahjongAI\.venv\Lib\site-packages\torchrl\envs\libs\pettingzoo.py:281: UserWarning: PettingZoo in TorchRL is tested using version == 1.24.3 , If you are using a different version and are experiencing compatibility issues,please raise an issue in the TorchRL github.
  warnings.warn(


In [ ]:
class CastToFloat(nn.Module):
    def forward(self, x):
        return x.float()   # or .to(torch.float32)

policy_net = nn.Sequential(
    nn.Flatten(-2),
    CastToFloat(),
    MultiAgentMLP(
        n_agent_inputs = 29 * 34,       
        n_agent_outputs = 75,       
        n_agents = 4,
        centralized = False,        
        share_params = True,      
        depth = 8,               
        num_cells = 1024,
        activation_class=torch.nn.Tanh
    )
)

policy_module = TensorDictModule(
    policy_net,
    in_keys=[("agents", "observation", "observation")],
    out_keys=[("agents", "logits")],
)

policy = ProbabilisticActor(
    module=policy_module,
    spec=env.action_spec_unbatched,
    in_keys={
        'logits': ('agents', 'logits'),
        'mask': ('agents', 'action_mask')
    }, # type: ignore
    out_keys=[env.action_key],
    distribution_class=MaskedCategorical,
    return_log_prob=True
)  # we'll need the log-prob for the PPO loss


c:\Users\ctc73\PycharmProjects\MahjongAI\.venv\Lib\site-packages\tensordict\_td.py:612: FutureWarning: TensorDict.to_module() is replacing an existing nn.Parameter in the destination module with a tensor leaf that is not an nn.Parameter. This historical behavior can remove the key from module.state_dict(). In tensordict v0.14, to_module() will preserve existing module parameter and buffer registrations by default. Pass preserve_module_state=False to keep the current replacement behavior, or preserve_module_state=True to opt in to the v0.14 behavior now.
  local_out = _set_tensor_dict(


In [ ]:
critic_net = nn.Sequential(
    nn.Flatten(-2),                    # [248,46] -> [248*46]
    CastToFloat(),
    nn.Linear(32*34, 512),
    nn.ReLU(),
    nn.Linear(512, 4),                # output 4 values (one per agent)
    nn.ReLU(),
    nn.Linear(512, 4),                # output 4 values (one per agent)
    nn.ReLU(),
    nn.Linear(512, 4),                # output 4 values (one per agent)
    nn.Unflatten(-1, (4, 1))          # reshape from [4] to [4,1]
)

critic = TensorDictModule(
    module=critic_net,
    in_keys=["state"],               # global state
    out_keys=[("agents", "state_value")],  # shape [4] values under agents
)

In [31]:
print("Running policy:", policy(env.reset()))
print("Running value:", critic(env.reset()))

Running policy: TensorDict(
    fields={
        agents: TensorDict(
            fields={
                action: Tensor(shape=torch.Size([4]), device=cpu, dtype=torch.int64, is_shared=False),
                action_log_prob: Tensor(shape=torch.Size([4]), device=cpu, dtype=torch.float32, is_shared=False),
                action_mask: Tensor(shape=torch.Size([4, 75]), device=cpu, dtype=torch.bool, is_shared=False),
                done: Tensor(shape=torch.Size([4, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                logits: Tensor(shape=torch.Size([4, 75]), device=cpu, dtype=torch.float32, is_shared=False),
                mask: Tensor(shape=torch.Size([4]), device=cpu, dtype=torch.bool, is_shared=False),
                observation: TensorDict(
                    fields={
                        observation: Tensor(shape=torch.Size([4, 29, 34]), device=cpu, dtype=torch.uint8, is_shared=False)},
                    batch_size=torch.Size([4]),
                    device=

In [32]:
# import pygame
# from pygame_visualizer import render_game_state
# env.rollout(256, policy=policy)
# pygame.init()
# screen = pygame.display.set_mode(size=(800, 800))
# font = pygame.font.Font("C:/Windows/Fonts/seguisym.ttf", 48)
# from sys import exit
# while True:
#     for event in pygame.event.get():
#         if event.type == pygame.QUIT:
#             pygame.quit()
#             exit()
#     screen.fill('white')
#     render_game_state(env._env.gamestate, screen, font)
#     pygame.display.update()


In [33]:
loss_module = ClipPPOLoss(
    actor_network=policy,
    critic_network=critic
)
loss_module.set_keys(  # We have to tell the loss where to find the keys
    reward=env.reward_key,
    action=env.action_key,
    value=("agents", "state_value"),
    # These last 2 keys will be expanded to match the reward shape
    done=("agents", "done"),                # per-agent
    terminated=("agents", "terminated"),
)
gamma = 0.995  # discount factor
lmbda = 0.9  # lambda for generalised advantage estimation
lr = 3e-4
loss_module.make_value_estimator(
    ValueEstimators.GAE, gamma=gamma, lmbda=lmbda
)  
GAE = loss_module.value_estimator

optim = torch.optim.Adam(loss_module.parameters(), lr)

loss_module = loss_module.to(device)

In [ ]:
num_epochs = 10
max_grad_norm = 1
frames_per_batch = 20000  # Number of team frames collected per training iteration
n_iters = 10  # Number of sampling and training iterations
total_frames = frames_per_batch * n_iters
minibatch_size = 32

while True:
    collector = Collector(
        env,
        policy,
        device='cpu',
        storing_device=device,
        frames_per_batch=frames_per_batch,
        total_frames=total_frames
    )
    replay_buffer = ReplayBuffer(
        storage=LazyTensorStorage(
            frames_per_batch, device=device
        ),  # We store the frames_per_batch collected at each iteration
        sampler=SamplerWithoutReplacement(),
        batch_size=minibatch_size,  # We will sample minibatches of this siz
    )

    from tqdm.auto import tqdm
    for tensordict_data in tqdm(collector):
        tensordict_data.set(
            ("next", "agents", "done"),
            tensordict_data.get(("next", "done"))
            .unsqueeze(-1)
            .expand(tensordict_data.get_item_shape(("next", env.reward_key))),
        )
        tensordict_data.set(
            ("next", "agents", "terminated"),
            tensordict_data.get(("next", "terminated"))
            .unsqueeze(-1)
            .expand(tensordict_data.get_item_shape(("next", env.reward_key))),
        )
        # We need to expand the done and terminated to match the reward shape (this is expected by the value estimator)

        with torch.no_grad():
            GAE(
                tensordict_data,
                params=loss_module.critic_network_params,
                target_params=loss_module.target_critic_network_params,
            )  # Compute GAE and add it to the data

        data_view = tensordict_data.reshape(-1)  # Flatten the batch size to shuffle data
        replay_buffer.extend(data_view)

        for _ in range(num_epochs):
            for _ in range(frames_per_batch // minibatch_size):
                subdata = replay_buffer.sample()
                subdata = subdata.to(device)
                loss_vals = loss_module(subdata)

                loss_value = (
                    loss_vals["loss_objective"]
                    + loss_vals["loss_critic"]
                    + loss_vals["loss_entropy"]
                )

                loss_value.backward()

                torch.nn.utils.clip_grad_norm_(
                    loss_module.parameters(), max_grad_norm
                )  # Optional

                optim.step()
                optim.zero_grad()

        collector.update_policy_weights_()

    # After training
    torch.save(policy_net.state_dict(), "actor_net.pth")
    torch.save(critic.state_dict(), "mahjong_critic.pth")

C:\Users\ctc73\AppData\Local\Temp\ipykernel_6284\4173163291.py:8: FutureWarning: The env passed to Collector is missing transforms required by the policy (InitTracker). From torchrl v0.15 the collector will append them automatically. To enable that behavior now (and silence this warning), pass `auto_register_policy_transforms=True`. To opt out permanently, pass `auto_register_policy_transforms=False`.
  collector = Collector(


2026-07-21 16:22:14,654 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([10000]) shape [END]


100%|██████████| 1/1 [02:43<00:00, 163.80s/it]
